In [14]:
import pickle
import string
import pandas as pd
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import SGDClassifier , LogisticRegression
from sklearn.naive_bayes import MultinomialNB , GaussianNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report , accuracy_score , confusion_matrix


lemmatize = WordNetLemmatizer()
stop_word = set(stopwords.words("english"))

cv = CountVectorizer()
model = LogisticRegression()


def remove_punctuation(text):
    new_text = text
    import string
    for i in new_text:
        if i in string.punctuation:
            new_text = new_text.replace(i , "")
    return new_text
    
def lower(text):
    return str(text).lower()


def tokenize(text):
    return word_tokenize(text)

def remove_digits(ls):
    f_list = []
    for i in ls:
        if i.isalpha():
            f_list.append(i)
    return f_list


def remove_stopwords(ls):
    f_list = []
    for i in ls:
        if i not in stopwords.words("english"):
            f_list.append(i)
    return f_list


def lemmat(ls):
    f_list = []
    for i in ls:
        a = lemmatize.lemmatize(i , pos="v")
        f_list.append(a)
    return f_list 


def join_words(ls):
    return " ".join(ls)


def preprocess(text):
    text = remove_punctuation(text)
    text = lower(text)
    text = tokenize(text)
    text = remove_digits(text)
    text = remove_stopwords(text)
    text = lemmat(text)
    text = join_words(text)
    return text


def train_model(x , y):
    x = x.apply(preprocess)
    x_vec = cv.fit_transform(x)
    model.fit(x_vec , y)

def predict_message(msg):
    msg = preprocess(msg)
    v = cv.transform([msg])
    result = model.predict(v)[0]
    if result == "spam":
        return "SPAM MESSAGE"
    else:
        return "NORMAL MESSAGE"

def save_model():
    pickle.dump(model , open("spam_ham_project.pkl" , "wb"))
    pickle.dump(cv , open("vectorized.pkl" , "wb"))

def load_model():
    global model , cv
    model = pickle.load(open("spam_ham_project.pkl" , "rb"))  
    cv = pickle.load(open("vectorized.pkl" , "rb")) 
  
data = pd.read_csv("spam.csv")
data.drop("Unnamed: 0" , axis=1 , inplace=True)
x = data["Messages"]
y = data["Result"]
x_train , x_test , y_train , y_test = train_test_split(x,y,test_size=0.2 , random_state=42)
train_model(x_train , y_train)
x_test = x_test.apply(preprocess)
x_test_vec = cv.transform(x_test)
y_pred = model.predict(x_test_vec)
print(accuracy_score(y_test , y_pred))
print(classification_report(y_test , y_pred))
print(confusion_matrix(y_test , y_pred))
save_model()

0.9856502242152466
              precision    recall  f1-score   support

         ham       0.98      1.00      0.99       954
        spam       1.00      0.90      0.95       161

    accuracy                           0.99      1115
   macro avg       0.99      0.95      0.97      1115
weighted avg       0.99      0.99      0.99      1115

[[954   0]
 [ 16 145]]


In [15]:
predict_message("congratulations")

'NORMAL MESSAGE'

In [16]:
predict_message("free won cash prize")

'SPAM MESSAGE'

In [17]:
predict_message("hey how are you")

'NORMAL MESSAGE'

In [18]:
predict_message("urgent prize claim")

'SPAM MESSAGE'

In [19]:
predict_message("free entry now win prize claim")

'SPAM MESSAGE'

In [20]:
predict_message("hi @win cash prize urgent")

'SPAM MESSAGE'

In [21]:
predict_message("hii its great to have you in meeting")

'NORMAL MESSAGE'

In [22]:
predict_message("winning prizes and cash")

'SPAM MESSAGE'

In [23]:
predict_message("winning prizes")   # km words h eslie 

'NORMAL MESSAGE'

In [24]:
predict_message("hii its great to have you in meeting n you will definetly win cash in this game")

'NORMAL MESSAGE'

In [25]:
predict_message("hii its great to have you in meeting and you will definetly win cash and prizes in this game")

'SPAM MESSAGE'